# train_meta.ipynb — MLメタモデル週次学習（Wave R5-1 / S7）

**実行環境**: Google Colab（無料CPU枠・週次手動）

**設計正本**: 要件定義書_統合改修_20260705.md §R5-1（変更禁止事項あり）

手順:
1. GDriveをマウントし `features.jsonl`（日次バッチが蓄積）を読み込む
2. yfinanceで20日先の対TOPIX超過リターンをラベル付け（`forward_excess_returns`＝リーク防止済み実装を共用）
3. LightGBM回帰 × **Purged Walk-Forward CV**（embargo 20営業日）
4. fold平均 rank IC と t統計量を算出
5. **昇格基準（変更禁止）: OOS rank IC >= 0.02 かつ t >= 2.0** を満たす場合のみ補正テーブルJSONをエクスポート
6. エクスポート後も **健太が `ML_META_ENABLED=True` にするまで日次適用されない**

In [ ]:
# ① セットアップ（Colab）
!pip -q install lightgbm yfinance
from google.colab import drive
drive.mount('/content/drive')

REPO = '/content/drive/MyDrive/stock_analyzer'   # ←環境に合わせて変更
import sys; sys.path.insert(0, REPO)
DATASET = f'{REPO}/data/ml_dataset/features.jsonl'
EXPORT  = f'{REPO}/data/ml_dataset/correction_table.json'
BENCH   = '1306.T'   # TOPIX連動ETF（^TPXは取得不安定のためETFで代用）
HORIZON = 20; EMBARGO = 20; N_FOLDS = 5; N_DECILES = 10; MAX_PT = 10.0
PROMOTE_IC = 0.02; PROMOTE_T = 2.0   # 変更禁止（§R5-1）

In [ ]:
# ② データセット読込
import json, pandas as pd
rows = [json.loads(l) for l in open(DATASET, encoding='utf-8') if l.strip()]
df = pd.DataFrame(rows)
print(f'{len(df)}行 / {df.date.nunique()}日 / {df.ticker.nunique()}銘柄')
feature_cols = [c for c in df.columns if c.startswith('f_')]
print('特徴量:', len(feature_cols), 'カテゴリ')

In [ ]:
# ③ ラベル付け: 20日先 対TOPIX超過リターン（リーク防止実装を本体と共用）
import yfinance as yf
from ml.dataset_builder import forward_excess_returns

start = df.date.min(); end = None
bench_px = yf.download(BENCH, start=start, auto_adjust=True, progress=False)['Close']
bench_by_date = {d.strftime('%Y-%m-%d'): float(v) for d, v in bench_px.dropna().items()}

targets = {}
for tk in df.ticker.unique():
    try:
        px = yf.download(tk, start=start, auto_adjust=True, progress=False)['Close'].dropna()
    except Exception:
        continue
    p = {d.strftime('%Y-%m-%d'): float(v) for d, v in px.items()}
    dates = sorted(set(p) & set(bench_by_date))
    ex = forward_excess_returns(dates, p, bench_by_date, HORIZON)
    for d, v in ex.items():
        targets[(d, tk)] = v

df['target'] = df.apply(lambda r: targets.get((r['date'], r['ticker'])), axis=1)
df = df.dropna(subset=['target']).reset_index(drop=True)
print(f'ラベル付与後: {len(df)}行（未来{HORIZON}日が確定した行のみ）')

In [ ]:
# ④ 前処理: レジーム4象限one-hot・size_tier・業種（要件§R5-1どおり）
X = df[feature_cols].fillna(50.0)
for q in ['A', 'B', 'C', 'D']:
    X[f'regime_{q}'] = (df['regime'] == q).astype(int)
tier_map = {'Large': 4, 'Mid': 3, 'Small': 2, 'Micro': 1, 'Nano': 0}
X['size_tier'] = df['size_tier'].map(tier_map).fillna(-1)
X['sector'] = df['sector'].astype('category').cat.codes
y = df['target']
print(X.shape)

In [ ]:
# ⑤ LightGBM × Purged Walk-Forward CV（本体の purged_cv を共用）
import lightgbm as lgb, numpy as np
from ml.purged_cv import purged_walk_forward_splits, rank_ic, ic_t_stat

dates_sorted = sorted(df.date.unique())
splits = purged_walk_forward_splits(dates_sorted, N_FOLDS, EMBARGO, HORIZON)
print(f'{len(splits)} folds（データ{len(dates_sorted)}日）')

fold_ics, models = [], []
for k, (tr_dates, te_dates) in enumerate(splits):
    tr = df.date.isin(tr_dates); te = df.date.isin(te_dates)
    if tr.sum() < 50 or te.sum() < 20:
        print(f'fold{k}: データ不足でスキップ'); continue
    m = lgb.LGBMRegressor(n_estimators=300, num_leaves=31,
                          learning_rate=0.05, min_child_samples=20,
                          random_state=42, verbose=-1)
    m.fit(X[tr], y[tr])
    # rank ICは日次クロスセクションで計算→fold内平均
    te_df = df[te].copy(); te_df['pred'] = m.predict(X[te])
    daily = [rank_ic(list(g['pred']), list(g['target']))
             for _, g in te_df.groupby('date') if len(g) >= 5]
    ic = float(np.mean(daily)) if daily else 0.0
    fold_ics.append(ic); models.append(m)
    print(f'fold{k}: rank IC = {ic:+.4f}（{len(daily)}日平均）')

oos_ic = float(np.mean(fold_ics)) if fold_ics else 0.0
t_stat = ic_t_stat(fold_ics)
print(f'\nOOS rank IC = {oos_ic:+.4f} / t = {t_stat:.2f}')
print('昇格判定:', '✅ 合格' if (oos_ic >= PROMOTE_IC and t_stat >= PROMOTE_T) else '❌ 不合格（エクスポートしない）')

In [ ]:
# ⑥ embargo有無の比較（受入条件: embargoでICが変わることの確認）
splits_no_embargo = purged_walk_forward_splits(dates_sorted, N_FOLDS, 0, 0)
ics_ne = []
for tr_dates, te_dates in splits_no_embargo:
    tr = df.date.isin(tr_dates); te = df.date.isin(te_dates)
    if tr.sum() < 50 or te.sum() < 20: continue
    m = lgb.LGBMRegressor(n_estimators=300, num_leaves=31, learning_rate=0.05,
                          min_child_samples=20, random_state=42, verbose=-1)
    m.fit(X[tr], y[tr])
    te_df = df[te].copy(); te_df['pred'] = m.predict(X[te])
    daily = [rank_ic(list(g['pred']), list(g['target']))
             for _, g in te_df.groupby('date') if len(g) >= 5]
    if daily: ics_ne.append(float(np.mean(daily)))
ic_ne = float(np.mean(ics_ne)) if ics_ne else 0.0
print(f'embargoなしIC = {ic_ne:+.4f} vs embargoありIC = {oos_ic:+.4f}')
print('→ embargoなしの方が高い場合、その差はリークによる過大評価分（正常な現象）')

In [ ]:
# ⑦ 補正テーブルのエクスポート（昇格基準を満たす場合のみ）
from datetime import date

if oos_ic >= PROMOTE_IC and t_stat >= PROMOTE_T and models:
    # 全データで最終モデル再学習
    final = lgb.LGBMRegressor(n_estimators=300, num_leaves=31, learning_rate=0.05,
                              min_child_samples=20, random_state=42, verbose=-1)
    final.fit(X, y)
    df2 = df.copy(); df2['pred'] = final.predict(X)
    df2['decile'] = pd.qcut(df2['pred'].rank(method='first'), N_DECILES, labels=False)
    # decile別の平均超過リターン → composite加減点へ線形スケール（上限±MAX_PT）
    dec_mean = df2.groupby('decile')['target'].mean()
    scale = MAX_PT / max(abs(dec_mean.min()), abs(dec_mean.max()), 1e-9)
    adjustments = {str(int(d)): round(float(min(MAX_PT, max(-MAX_PT, v * scale))), 1)
                   for d, v in dec_mean.items()}
    # 日次適用用の線形近似係数（LightGBM本体をGitHub Actionsに持ち込まないための軽量化）
    from sklearn.linear_model import Ridge
    lin = Ridge(alpha=1.0).fit(X, final.predict(X))
    coefs = {'_intercept': float(lin.intercept_)}
    coefs.update({c: float(w) for c, w in zip(X.columns, lin.coef_)})
    table = {
        'trained_at': date.today().isoformat(),
        'oos_rank_ic': round(oos_ic, 4), 't_stat': round(t_stat, 2),
        'n_folds': len(fold_ics), 'feature_names': list(X.columns),
        'decile_adjustments': adjustments, 'model_coefs': coefs,
    }
    json.dump(table, open(EXPORT, 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
    print(f'✅ エクスポート: {EXPORT}')
    print('decile補正:', adjustments)
    print('\n※ 日次適用には健太が config.ML_META_ENABLED=True へ変更（承認必須）')
else:
    print('❌ 昇格基準未達（IC>=0.02 かつ t>=2.0）— エクスポートせず終了')